# 📊 Thực Nghiệm 03: So Sánh Các Mô Hình Baseline Phát Hiện Gian Lận
**Tác giả**: Phan Võ Minh Đồng (DH52200514) | **Mục tiêu**: So sánh Logistic Regression (E01), Random Forest (E02), XGBoost (E03).


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_curve, roc_curve, auc, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')
print('-> Đã nạp thư viện.')


### 2. Nạp dữ liệu và EDA


In [ ]:
np.random.seed(42); n_samples = 40000; n_fraud = int(n_samples * 0.008); n_norm = n_samples - n_fraud
X_norm = np.column_stack([np.random.poisson(1.2, n_norm), np.random.poisson(3.0, n_norm), np.random.exponential(5e5, n_norm), np.random.poisson(1.0, n_norm), np.random.uniform(1e6, 5e6, n_norm), np.random.uniform(0.5, 1.8, n_norm), np.random.binomial(1, 0.04, n_norm), np.random.binomial(1, 0.08, n_norm)])
X_fr = np.column_stack([np.random.poisson(5.5, n_fraud), np.random.poisson(14.0, n_fraud), np.random.exponential(6e6, n_fraud), np.random.poisson(4.2, n_fraud), np.random.uniform(5e5, 2e6, n_fraud), np.random.uniform(4.0, 15.0, n_fraud), np.random.binomial(1, 0.75, n_fraud), np.random.binomial(1, 0.60, n_fraud)])
X = np.vstack([X_norm, X_fr]); y = np.concatenate([np.zeros(n_norm), np.ones(n_fraud)])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler(); X_tr_s = scaler.fit_transform(X_tr); X_te_s = scaler.transform(X_te)
print(f'Train: {len(y_tr)} | Test: {len(y_te)}')


### 3. Huấn luyện và Đánh giá


In [ ]:
models = {
    'Logistic Regression (E01)': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest (E02)': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost (E03)': XGBClassifier(n_estimators=150, max_depth=5, scale_pos_weight=(len(y_tr)-y_tr.sum())/y_tr.sum(), eval_metric='aucpr', random_state=42),
}
preds, metrics = {}, []
for name, m in models.items():
    m.fit(X_tr_s, y_tr); prob = m.predict_proba(X_te_s)[:, 1]; preds[name] = prob
    p, r, _ = precision_recall_curve(y_te, prob); fpr, tpr, _ = roc_curve(y_te, prob)
    metrics.append({'Model': name, 'PR-AUC': round(auc(r, p), 4), 'ROC-AUC': round(auc(fpr, tpr), 4)})
pd.DataFrame(metrics)


### 4. Đồ thị so sánh PR Curve


In [ ]:
plt.figure(figsize=(8, 5), dpi=150)
for name, prob in preds.items():
    p, r, _ = precision_recall_curve(y_te, prob)
    plt.plot(r, p, lw=2, label=f'{name} (PR-AUC={auc(r, p):.3f})')
plt.title('Precision-Recall Curve Comparison', fontweight='bold'); plt.xlabel('Recall'); plt.ylabel('Precision'); plt.legend(); plt.show()
